In [1]:
%%capture
import os

# Clona la repository ufficiale se non esiste
if not os.path.exists('ControlNet2026'):
    !git clone https://github.com/Salvatorevil03/ControlNet2026

%cd ControlNet2026

!pip install -q gradio pytorch-lightning omegaconf einops test-tube streamlit streamlit-drawable-canvas webdataset kornia open_clip_torch invisible-watermark torchmetrics timm addict yapf basicsr imageio-ffmpeg opencv-contrib-python

In [2]:
# @title ⬇️ Selezione Modelli ControlNet 1.5
# @markdown Spunta i modelli che desideri scaricare (attenzione: ogni modello pesa ~5.7 GB).

import os

# --- INIZIO FORM COLAB ---
Canny = True # @param {type:"boolean"}
Depth = False # @param {type:"boolean"}
HED = False # @param {type:"boolean"}
MLSD = False # @param {type:"boolean"}
Normal = False # @param {type:"boolean"}
OpenPose = False # @param {type:"boolean"}
Scribble = False # @param {type:"boolean"}
Segmentation = False # @param {type:"boolean"}
# --- FINE FORM COLAB ---

# Creiamo la lista dei modelli da scaricare in base alle tue spunte
models_to_download = []
if Canny: models_to_download.append("control_sd15_canny.pth")
if Depth: models_to_download.append("control_sd15_depth.pth")
if HED: models_to_download.append("control_sd15_hed.pth")
if MLSD: models_to_download.append("control_sd15_mlsd.pth")
if Normal: models_to_download.append("control_sd15_normal.pth")
if OpenPose: models_to_download.append("control_sd15_openpose.pth")
if Scribble: models_to_download.append("control_sd15_scribble.pth")
if Segmentation: models_to_download.append("control_sd15_seg.pth")

base_url = "https://huggingface.co/lllyasviel/ControlNet/resolve/main/models/"
model_dir = "/content/ControlNet2026/models"
 #/kaggle/working/ControlNet2026/models if running on Kaggle
# Assicuriamoci che la cartella esista
os.makedirs(model_dir, exist_ok=True)

if not models_to_download:
    print("⚠️ Nessun modello selezionato. Spunta almeno una casella.")
else:
    print(f"🚀 Inizio controllo e download di {len(models_to_download)} modelli...\n")

    for model in models_to_download:
        model_path = os.path.join(model_dir, model)
        if not os.path.exists(model_path):
            print(f"⬇️ Scaricando {model}...")
            # Scarica in modo silenzioso mostrando solo la progress bar
            !wget -q --show-progress -O "{model_path}" "{base_url}{model}"
        else:
            print(f"✅ {model} già presente, salto il download.")

    print("\n🎉 Download completato!")

🚀 Inizio controllo e download di 1 modelli...

⬇️ Scaricando control_sd15_canny.pth...
  /content/ControlN   4%[                    ] 245.38M   152MB/s               ^C

🎉 Download completato!


In [4]:
import os
import sys

"""
--- PATCH BASICSR vs TORCHVISION (V2) ---
Aggiorna l'import senza tentare di caricare la libreria rotta!
"""

print("Cerco il file difettoso di basicsr...")

# Troviamo la cartella dist-packages senza importare basicsr
dist_packages = [p for p in sys.path if 'dist-packages' in p][0]
degradations_file = os.path.join(dist_packages, 'basicsr', 'data', 'degradations.py')
print(degradations_file)

if os.path.exists(degradations_file):
    with open(degradations_file, 'r', encoding='utf-8') as f:
        content = f.read()

    vecchio_import = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    nuovo_import = "from torchvision.transforms.functional import rgb_to_grayscale"

    if vecchio_import in content:
        content = content.replace(vecchio_import, nuovo_import)

        with open(degradations_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ Patch applicata con successo! Libreria basicsr svecchiata.")
    else:
        print("⚠️ Patch già applicata o stringa non trovata.")
else:
    print(f"❌ Impossibile trovare il file: {degradations_file}")

Cerco il file difettoso di basicsr...
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py


In [3]:
"""
--- PATCH PYTORCH LIGHTNING 1.x -> 2.x ---
Risolve il `ModuleNotFoundError` sostituendo il vecchio modulo
`utilities.distributed` con il nuovo `utilities.rank_zero` in tutti i file .py.
Permette di usare PL 2.x nativo di Colab senza fare downgrade.

import os

print("Inizio patch per compatibilità PyTorch Lightning 2.x...")

# Percorso principale della repo
repo_dir = '/content/ControlNet2026'

# Sostituzione dinamica in tutti i file Python
files_patched = 0
for root, dirs, files in os.walk(repo_dir):
    for file in files:
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()

            # Se trova il vecchio modulo, lo aggiorna al nuovo percorso
            if 'pytorch_lightning.utilities.distributed' in content:
                content = content.replace('pytorch_lightning.utilities.distributed', 'pytorch_lightning.utilities.rank_zero')
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(content)
                print(f"✅ Fix applicato a: {filepath.replace(repo_dir, '')}")
                files_patched += 1

if files_patched == 0:
    print("Nessun file aveva bisogno del fix (già applicato o repo modificata).")
else:
    print("Patch completata con successo! Ora puoi avviare l'inferenza.")
"""

Inizio patch per compatibilità PyTorch Lightning 2.x...
Nessun file aveva bisogno del fix (già applicato o repo modificata).


In [ ]:
import os
import sys
import pytorch_lightning

# Fix per PyTorch Lightning 2.x
pl_path = os.path.dirname(pytorch_lightning.__file__)
dist_file = os.path.join(pl_path, 'utilities', 'distributed.py')

if not os.path.exists(dist_file):
    print("Applicazione patch di compatibilità per PyTorch Lightning...")
    os.makedirs(os.path.dirname(dist_file), exist_ok=True)
    with open(dist_file, 'w') as f:
        f.write('from pytorch_lightning.utilities.rank_zero import *')
    print("✅ Patch applicata.")

# Avvio del comando originale
!python gradio_canny2image.py

Applicazione patch di compatibilità per PyTorch Lightning...
✅ Patch applicata.
logging improved.
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
tokenizer_config.json: 100% 905/905 [00:00<00:00, 5.13MB/s]
vocab.json: 961kB [00:00, 50.8MB/s]
merges.txt: 525kB [00:00, 121MB/s]
tokenizer.json: 2.22MB [00:00, 115MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 2.13MB/s]
config.json: 4.52kB [00:00, 11.4MB/s]
model.safetensors: 100% 1.71G/1.71G [00:20<00:00, 82.9MB/s]
Loading weights: 100% 196/196 [00:00<00:00, 1182.94it/s, Materializing param=text_model.final_layer_norm.weight]
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_canny.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running 

In [ ]:
!python gradio_pose2image.py

logging improved.
Downloading: "https://huggingface.co/lllyasviel/ControlNet/resolve/main/annotator/ckpts/body_pose_model.pth" to /content/ControlNet2026/annotator/ckpts/body_pose_model.pth

100% 200M/200M [00:00<00:00, 228MB/s]
Downloading: "https://huggingface.co/lllyasviel/ControlNet/resolve/main/annotator/ckpts/hand_pose_model.pth" to /content/ControlNet2026/annotator/ckpts/hand_pose_model.pth

100% 141M/141M [00:01<00:00, 90.0MB/s]
cuda
cuda
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
tokenizer_config.json: 100% 905/905 [00:00<00:00, 4.36MB/s]
vocab.json: 961kB [00:00, 39.2MB/s]
merges.txt: 525kB [00:00, 105MB/s]
tokenizer.json: 2.22MB [00:00, 130MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 1.98MB/s]
config.json: 4.52kB 

In [ ]:
!python gradio_scribble2image_interactive.py

logging improved.
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 969.88it/s, Materializing param=text_model.final_layer_norm.weight] 
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_scribble.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://514b02c3af61535e4a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Seed set to 1046977011
Data shape for DDIM sampling is (1, 4, 64, 64), eta 0
Running DDIM Sampling with 20 timesteps
DDIM Sampler: 100% 20/20 [00